# Multi-day pipeline

Batched version of `components.ipynb`. Runs all three exit classifiers (v1, v2, v3) and the hive-return classifier across **every cached `(date, system_id)` pair** and writes two CSVs that downstream notebooks can consume:

- **`daily_summary.csv`** - one row per `(date, system_id)`. Counts (`n_tracks`, `n_returns`, `n_v1`, `n_v2`, `n_v3`), `re_ratio_v3`, and aggregate trip-duration stats.
- **`per_track_indicators.csv`** - one row per flight track with timestamps and per-track features (`hive_return`, `hive_exit_v3`, `tortuosity`, `v2_reason`, ...). This is what `exposure_analysis.ipynb` uses for distribution-level comparisons (box plots, histograms, statistical tests).

## Prerequisites

Run `components.ipynb` once for each date you want to include. That populates `cache/flight_data_<DATE>_system_<ID>/` with `detections.csv` and a `tracks/` folder. This notebook then reads from that cache - no API calls, no re-downloading.

If you add a new date later, run components for that date and re-run this notebook. It scans the cache fresh each time.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


## 1. Configuration

`DATES` filters which dates to process. Set to `None` to include every cached date.

`HIVE_BY_SYSTEM` is the calibrated hive position per camera (from `calibration_and_validation.ipynb`, section 3). Update if your camera moves.


In [2]:
# --- Date filter ---------------------------------------------------------
DATES = None  # None = all cached dates; or e.g. pd.date_range("2026-04-19","2026-05-03")
SYSTEM_IDS = [900, 939]

# --- Hive positions per camera (from hive-calibration cell) -------------
HIVE_BY_SYSTEM = {
    900: np.array([-0.04,  -0.665, -1.195]),
    939: np.array([-0.086, -0.828, -1.045]),
}

# --- Classifier thresholds (must match components.ipynb) ----------------
TOL          = 0.10   # m, return + v1-exit tolerance
TAIL_FRAMES  = 10     # frames at end of track to inspect for return
HEAD_FRAMES  = 5      # frames at start to inspect for exit / fit velocity
EXIT_TOL_V2  = 0.20   # m, v2 closest-approach tolerance
MIN_SPEED    = 0.5    # m/s, v2 minimum initial speed
MAX_LAG_S    = 2.0    # s, v2 maximum back-extrapolation lag
FPS          = 60.0   # PATS-C frame rate

# --- Paths ---------------------------------------------------------------
# Notebook expected to run from .../pats/  (same cwd as components.ipynb)
CACHE_BASE = Path("cache")
OUT_DIR    = Path("data/multi_day")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Cache : {CACHE_BASE.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")


Cache : /Users/jaspe/Projects/Claude/Bumblebee-monitoring/src/flight_analysis/pats/cache
Output: /Users/jaspe/Projects/Claude/Bumblebee-monitoring/src/flight_analysis/pats/data/multi_day


## 2. Classifier helpers

Self-contained versions of the classifiers and indicators. Same logic as `components.ipynb`, just packaged into pure functions so the loop below can call them per track.


In [3]:
STRONG_NO = {"closest_in_future", "too_far", "too_much_lag"}
WEAK_NO   = {"slow", "short_track"}


def valid_xyz(trk):
    """(N, 3) array of valid positions, or None if there aren't any."""
    if "pos_valid_insect" in trk.columns:
        trk = trk[trk["pos_valid_insect"] == 1]
    if len(trk) == 0:
        return None
    return trk[["posX_insect", "posY_insect", "posZ_insect"]].to_numpy(dtype=float)


def is_hive_return(xyz, hive, tol=TOL, tail=TAIL_FRAMES):
    if xyz is None or len(xyz) == 0:
        return False, np.nan
    tail_xyz = xyz[-tail:]
    d = float(np.min(np.linalg.norm(tail_xyz - hive, axis=1)))
    return (d <= tol), d


def is_hive_exit_v1(xyz, hive, tol=TOL, head=HEAD_FRAMES):
    if xyz is None or len(xyz) == 0:
        return False, np.nan
    head_xyz = xyz[:head]
    d = float(np.min(np.linalg.norm(head_xyz - hive, axis=1)))
    return (d <= tol), d


def is_hive_exit_v2(xyz, hive, head_n=HEAD_FRAMES, exit_tol=EXIT_TOL_V2,
                    min_speed=MIN_SPEED, max_lag_s=MAX_LAG_S, fps=FPS):
    if xyz is None or len(xyz) < head_n:
        return False, "short_track"
    head = xyz[:head_n]
    p0   = head[0]
    t = np.arange(head_n) / fps
    v = np.array([np.polyfit(t, head[:, k], 1)[0] for k in range(3)])
    speed = float(np.linalg.norm(v))
    if speed < min_speed:
        return False, "slow"
    rel    = p0 - hive
    t_star = -float(np.dot(v, rel)) / (speed ** 2)
    if t_star > 0:
        return False, "closest_in_future"
    if abs(t_star) > max_lag_s:
        return False, "too_much_lag"
    d_min = float(np.linalg.norm(p0 + v * t_star - hive))
    return (d_min <= exit_tol), ("ok" if d_min <= exit_tol else "too_far")


def path_tortuosity(xyz):
    """arc length / end-to-end displacement. >= 1.0; 1.0 means perfect line."""
    if xyz is None or len(xyz) < 2:
        return np.nan
    arc = float(np.linalg.norm(np.diff(xyz, axis=0), axis=1).sum())
    end_to_end = float(np.linalg.norm(xyz[-1] - xyz[0]))
    return np.nan if end_to_end < 1e-6 else arc / end_to_end


## 3. Process every cached `(date, system_id)` pair

For each cached pair, load detections + per-uid tracks, classify each track, and accumulate two tables:
- per-track rows (one row per flight)
- per-day rows (one row per date+system aggregate)

Trip duration is computed greedily within each day+camera: each v3 exit is matched to the next un-matched return after it.


In [4]:
pat = re.compile(r"flight_data_(\d{4}-\d{2}-\d{2})_system_(\d+)$")

target_dates = None
if DATES is not None:
    target_dates = {pd.Timestamp(d).date().isoformat() for d in DATES}

per_track_rows = []
daily_rows     = []

cached_pairs = sorted(CACHE_BASE.glob("flight_data_*_system_*"))
print(f"Found {len(cached_pairs)} cached folders")

for d in cached_pairs:
    m = pat.match(d.name)
    if not m:
        continue
    date_str, sys_str = m.groups()
    sys_id = int(sys_str)
    if sys_id not in SYSTEM_IDS:
        continue
    if target_dates is not None and date_str not in target_dates:
        continue
    if sys_id not in HIVE_BY_SYSTEM:
        print(f"  skipping {d.name} - no hive position for system {sys_id}")
        continue

    det_csv    = d / "detections.csv"
    tracks_dir = d / "tracks"
    if not det_csv.exists() or not tracks_dir.exists():
        continue

    dets = pd.read_csv(det_csv)
    dets["ts"] = pd.to_datetime(dets["datetime"],
                                format="%Y%m%d_%H%M%S", errors="coerce")

    hive = HIVE_BY_SYSTEM[sys_id]
    n_tracks = n_ret = n_v1 = n_v2 = n_v3 = 0
    exit_ts  = []
    return_ts = []

    for _, det in dets.iterrows():
        uid = int(det["uid"])
        trk_path = tracks_dir / f"{uid}.csv"
        if not trk_path.exists():
            continue
        trk = pd.read_csv(trk_path)
        xyz = valid_xyz(trk)
        if xyz is None or len(xyz) == 0:
            continue
        n_tracks += 1

        is_ret, _   = is_hive_return(xyz, hive)
        is_v1, _    = is_hive_exit_v1(xyz, hive)
        is_v2, rsn  = is_hive_exit_v2(xyz, hive)
        is_v3       = is_v2 or (is_v1 and rsn in WEAK_NO)
        tort        = path_tortuosity(xyz)

        if is_ret:
            n_ret += 1
            return_ts.append(det["ts"])
        if is_v1: n_v1 += 1
        if is_v2: n_v2 += 1
        if is_v3:
            n_v3 += 1
            exit_ts.append(det["ts"])

        per_track_rows.append({
            "date":          date_str,
            "system_id":     sys_id,
            "uid":           uid,
            "ts":            det["ts"],
            "n_samples":     len(xyz),
            "hive_return":   bool(is_ret),
            "hive_exit_v1":  bool(is_v1),
            "hive_exit_v2":  bool(is_v2),
            "hive_exit_v3":  bool(is_v3),
            "v2_reason":     rsn,
            "tortuosity":    tort,
        })

    # --- Trip duration (greedy match v3 exit -> next un-matched return) -
    ex_sorted = sorted(t for t in exit_ts   if pd.notna(t))
    rt_sorted = sorted(t for t in return_ts if pd.notna(t))
    used  = [False] * len(rt_sorted)
    trips = []
    for et in ex_sorted:
        for j, rt in enumerate(rt_sorted):
            if used[j] or rt <= et:
                continue
            trips.append((rt - et).total_seconds())
            used[j] = True
            break

    median_trip = float(np.median(trips)) if trips else np.nan
    mean_trip   = float(np.mean(trips))   if trips else np.nan

    daily_rows.append({
        "date":          date_str,
        "system_id":     sys_id,
        "n_tracks":      n_tracks,
        "n_returns":     n_ret,
        "n_v1":          n_v1,
        "n_v2":          n_v2,
        "n_v3":          n_v3,
        "re_ratio_v3":   (n_ret / n_v3) if n_v3 else np.nan,
        "median_trip_s": median_trip,
        "mean_trip_s":   mean_trip,
        "n_matched":     len(trips),
    })

    print(f"  {date_str}  sys {sys_id:3d}: "
          f"tracks={n_tracks:5d}  v3={n_v3:4d}  returns={n_ret:4d}  "
          f"trips={len(trips):4d}")

daily_summary = (pd.DataFrame(daily_rows)
                 .sort_values(["date", "system_id"])
                 .reset_index(drop=True))
per_track     = pd.DataFrame(per_track_rows)

print("\n=== summary ===")
print(daily_summary.to_string(index=False))


Found 43 cached folders
  2026-04-13  sys 900: tracks=  369  v3=  26  returns=  51  trips=  26
  2026-04-19  sys 900: tracks= 1080  v3= 152  returns= 226  trips= 152
  2026-04-20  sys 900: tracks= 1283  v3= 182  returns= 253  trips= 181
  2026-04-20  sys 939: tracks=  280  v3=   9  returns=   1  trips=   1
  2026-04-21  sys 900: tracks=   21  v3=   1  returns=   0  trips=   0
  2026-04-21  sys 939: tracks=    0  v3=   0  returns=   0  trips=   0
  2026-04-22  sys 900: tracks=  846  v3= 110  returns= 139  trips= 110
  2026-04-22  sys 939: tracks=  659  v3=  87  returns=  91  trips=  86
  2026-04-23  sys 900: tracks=  793  v3= 109  returns= 146  trips= 109
  2026-04-23  sys 939: tracks=  643  v3=  90  returns=  82  trips=  79
  2026-04-24  sys 900: tracks=  898  v3= 111  returns= 164  trips= 111
  2026-04-24  sys 939: tracks=  534  v3=  47  returns=  37  trips=  36
  2026-04-25  sys 900: tracks=  842  v3= 105  returns= 183  trips= 105
  2026-04-25  sys 939: tracks=  422  v3=  27  returns

## 4. Save outputs

Two CSVs in `data/multi_day/`. `exposure_analysis.ipynb` reads from here.


In [5]:
daily_summary.to_csv(OUT_DIR / "daily_summary.csv", index=False)
per_track.to_csv(OUT_DIR / "per_track_indicators.csv", index=False)

print(f"Wrote:")
print(f"  {OUT_DIR / 'daily_summary.csv'}        ({daily_summary.shape[0]} rows)")
print(f"  {OUT_DIR / 'per_track_indicators.csv'} ({per_track.shape[0]} rows)")


Wrote:
  data/multi_day/daily_summary.csv        (42 rows)
  data/multi_day/per_track_indicators.csv (33661 rows)
